# 08 · Food Relief Priority Index

Combine P(High ∪ Severe) with a food-fragility score. Fragility scalers are fit on TRAIN only, then applied to the test set. One folium map is rendered per TEST hurricane.

In [1]:
import sys, os
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns', 60)
from config import (
    DATA_PATHS, HURRICANE_META, STATES_IN_SCOPE,
    TARGET_COL, TARGET_CLASS_COL, FEATURE_GROUPS,
    RANDOM_STATE, SEVERITY_BINS, SEVERITY_LABELS,
)
RAW = DATA_PATHS['raw']; INTERIM = DATA_PATHS['interim']
PROC = DATA_PATHS['processed']; MODELS = DATA_PATHS['models']
OUT = DATA_PATHS['outputs']


In [2]:
import joblib, folium
from src.priority_index import fit_fragility_scalers, apply_fragility, priority_index, save_scalers
df = pd.read_csv(PROC / 'abt_with_clusters.csv', dtype={'zip_code': str}).dropna(subset=[TARGET_CLASS_COL])
pkg = joblib.load(MODELS / 'best_classifier.pkl')
pipe, le, features = pkg['pipe'], pkg['label_encoder'], pkg['features']

In [3]:
# Fit fragility scalers on TRAIN only
train = df[df['train_test_split']=='TRAIN']
scalers = fit_fragility_scalers(train)
save_scalers(scalers, MODELS / 'fragility_scalers.pkl')
df = apply_fragility(df, scalers)

In [4]:
# Predict probabilities for TEST set
te = df[df['train_test_split']=='TEST'].copy()
proba = pipe.predict_proba(te[features])
te_prio = priority_index(te, proba, list(le.classes_))
top = te_prio.sort_values('priority_rank').head(20)
top[['priority_rank','zip_code','state','hurricane_name',
     'food_relief_priority_index','prob_high_or_severe',
     'food_fragility_score','svi_overall','damage_severity_class']]

C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


,priority_rank,zip_code,state,hurricane_name,food_relief_priority_index,prob_high_or_severe,food_fragility_score,svi_overall,damage_severity_class
676,1,33438,FL,Ian,0.911225,0.995460,0.784873,0.979488,High
694,2,33476,FL,Ian,0.903042,0.992987,0.768124,0.998744,Severe
1592,3,31544,GA,Helene,0.901703,0.993295,0.764317,0.828251,Severe
410,4,32190,FL,Ian,0.899704,0.988425,0.766622,0.832009,High
898,5,33973,FL,Ian,0.898964,0.998957,0.748973,0.768853,Severe
384,6,32130,FL,Ian,0.898693,0.993013,0.757212,0.853312,High
2311,7,33834,FL,Milton,0.898608,0.996917,0.751143,0.796675,High
837,8,33854,FL,Ian,0.898294,0.991594,0.758343,0.732200,Low
1098,9,32096,FL,Idalia,0.896059,0.996374,0.745586,0.768848,Severe
821,10,33834,FL,Ian,0.895974,0.992527,0.751143,0.796675,Severe


In [5]:
te_prio.to_csv(OUT / 'priority_rankings.csv', index=False)
print('saved priority_rankings.csv', te_prio.shape)

saved priority_rankings.csv (2587, 73)


## Folium choropleth maps — one per TEST hurricane

In [6]:
import geopandas as gpd
zcta_path = next((RAW / 'zcta').rglob('*.shp'))
zcta = gpd.read_file(zcta_path).to_crs('EPSG:4326')
zcta['zip_code'] = zcta['ZCTA5CE20'].astype(str).str.zfill(5)

TOP_N = 200  # render only highest-priority ZIPs per hurricane

# Dynamically render a map for each TEST hurricane that has rows in te_prio.
# Use the centroid of the affected zips as the map center (no hardcoded coords).
test_hurricanes = sorted(te_prio['hurricane_name'].unique())

for hn in test_hurricanes:
    sub = (te_prio[te_prio['hurricane_name'] == hn]
           .groupby('zip_code', as_index=False)
           .agg(priority_index_norm=('priority_index_norm', 'max'),
                priority_rank=('priority_rank', 'min'),
                food_relief_priority_index=('food_relief_priority_index', 'max'),
                prob_high_or_severe=('prob_high_or_severe', 'max'),
                food_fragility_score=('food_fragility_score', 'max'))
           .nsmallest(TOP_N, 'priority_rank'))
    z = zcta.merge(sub, on='zip_code')
    if z.empty:
        print('no zips for', hn); continue
    z['geometry'] = z.geometry.simplify(tolerance=0.001, preserve_topology=True)
    cent = z.geometry.centroid.unary_union.centroid
    m = folium.Map(location=[cent.y, cent.x], zoom_start=7, tiles='cartodbpositron')
    folium.Choropleth(
        geo_data=z, data=z,
        columns=['zip_code', 'priority_index_norm'],
        key_on='feature.properties.zip_code',
        fill_color='YlOrRd', fill_opacity=0.7, line_opacity=0.2,
        legend_name=f'Priority index — {hn} (top {TOP_N})',
    ).add_to(m)
    folium.GeoJson(
        z, name='details',
        style_function=lambda _: {'fillOpacity': 0, 'color': 'transparent'},
        tooltip=folium.GeoJsonTooltip(
            fields=['zip_code', 'priority_rank', 'priority_index_norm',
                    'prob_high_or_severe', 'food_fragility_score'],
            aliases=['ZIP', 'Rank', 'Priority (norm)', 'P(High∪Severe)', 'Fragility'],
            localize=True),
    ).add_to(m)
    out = OUT / f'priority_map_{hn.lower()}.html'
    m.save(str(out)); print(f'saved {out}  ({len(z)} ZIPs)')

C:\Users\chait\AppData\Local\Temp\ipykernel_20452\1253067888.py:25: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  cent = z.geometry.centroid.unary_union.centroid
C:\Users\chait\AppData\Local\Temp\ipykernel_20452\1253067888.py:25: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  cent = z.geometry.centroid.unary_union.centroid


saved C:\Users\chait\OneDrive\Documents\ML_DATA_245\hurricane-food-relief\outputs\priority_map_helene.html  (200 ZIPs)


C:\Users\chait\AppData\Local\Temp\ipykernel_20452\1253067888.py:25: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  cent = z.geometry.centroid.unary_union.centroid
C:\Users\chait\AppData\Local\Temp\ipykernel_20452\1253067888.py:25: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  cent = z.geometry.centroid.unary_union.centroid


saved C:\Users\chait\OneDrive\Documents\ML_DATA_245\hurricane-food-relief\outputs\priority_map_ian.html  (200 ZIPs)


C:\Users\chait\AppData\Local\Temp\ipykernel_20452\1253067888.py:25: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  cent = z.geometry.centroid.unary_union.centroid
C:\Users\chait\AppData\Local\Temp\ipykernel_20452\1253067888.py:25: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  cent = z.geometry.centroid.unary_union.centroid


saved C:\Users\chait\OneDrive\Documents\ML_DATA_245\hurricane-food-relief\outputs\priority_map_ida.html  (200 ZIPs)


C:\Users\chait\AppData\Local\Temp\ipykernel_20452\1253067888.py:25: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  cent = z.geometry.centroid.unary_union.centroid
C:\Users\chait\AppData\Local\Temp\ipykernel_20452\1253067888.py:25: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  cent = z.geometry.centroid.unary_union.centroid


saved C:\Users\chait\OneDrive\Documents\ML_DATA_245\hurricane-food-relief\outputs\priority_map_idalia.html  (200 ZIPs)


C:\Users\chait\AppData\Local\Temp\ipykernel_20452\1253067888.py:25: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  cent = z.geometry.centroid.unary_union.centroid
C:\Users\chait\AppData\Local\Temp\ipykernel_20452\1253067888.py:25: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  cent = z.geometry.centroid.unary_union.centroid


saved C:\Users\chait\OneDrive\Documents\ML_DATA_245\hurricane-food-relief\outputs\priority_map_milton.html  (200 ZIPs)


## Top-50 vs bottom-50 summary

In [7]:
top50 = te_prio.nsmallest(50, 'priority_rank')
bot50 = te_prio.nlargest(50, 'priority_rank')
pd.DataFrame({
  'top50': [top50['food_desert_flag'].sum(), top50['svi_overall'].mean(),
            top50[TARGET_COL].mean()],
  'bot50': [bot50['food_desert_flag'].sum(), bot50['svi_overall'].mean(),
            bot50[TARGET_COL].mean()],
}, index=['food_desert_count','mean_svi','mean_damage_per_1k'])

,top50,bot50
food_desert_count,50.000000,0.000000
mean_svi,0.757939,0.357330
mean_damage_per_1k,69.241220,1.616381
